# Indonesian Batik Motifs Classification v3

This notebook implements the v3 architecture and training pipeline for the Indonesian Batik Motifs Detector.

### Major Architectural Fixes in v3:
1. **Double-Preprocessing Bug Resolved**: Fixed the dynamic range collapse where v2 normalized images twice ($[-1.008, -0.992]$). The model now has a single internal `Rescaling(scale=1/127.5, offset=-1.0)` layer and natively receives raw $[0, 255]$ float32 pixels.
2. **Perceptual Hash Deduplication**: Uses 64-bit dHash and Hamming distance to eliminate train/val/test data leakage from redundant duplicate images.
3. **Calibrated Loss & Capped Class Weighting**: Defaults to label-smoothed Categorical Crossentropy (`label_smoothing=0.05`) with capped Effective Number weights (`max_weight=3.0`), plus fixed true-class `p_t` Focal Loss ($\gamma = 2.0$).
4. **StratifiedGroupKFold 5-Fold Cross-Validation**: Two-phase fine-tuning (frozen backbone head training followed by top 120 layers unfreezing with cosine warmup and frozen BatchNormalization).
5. **Zero-Backend Export**: Automatic serialization to FP16 TFLite and ONNX for client-side in-browser WebAssembly execution.

## 1. Environment Configuration and Hyperparameters


In [ ]:
"""Cell 1: Environment detection, deterministic seeding, and hyperparameters."""

import os
import platform
import random
import warnings
from pathlib import Path

import numpy as np
import tensorflow as tf

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def auto_detect_dataset_dir() -> tuple[str, str]:
    """Detect dataset and output directories from env vars, Kaggle, Colab, or local workspace."""
    env_dataset = os.environ.get("BATIK_DATASET_DIR")
    env_output = os.environ.get("BATIK_OUTPUT_DIR", "./outputs/v3_train")
    if env_dataset and Path(env_dataset).exists():
        return env_dataset, env_output

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.rglob("*"):
            if p.is_dir() and (p / "Bali_Barong").exists():
                return str(p), "/kaggle/working"

    local_paths = [
        Path("/content/dataset"),
        Path("data/dataset"),
        Path("./dataset"),
    ]
    for lp in local_paths:
        if lp.exists() and (lp / "Bali_Barong").exists():
            return str(lp), env_output

    return "./dataset", env_output


DATASET_DIR, OUTPUT_DIR = auto_detect_dataset_dir()
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUTPUT_DIR, "best_batik_model_v3.keras")
TFLITE_PATH = os.path.join(OUTPUT_DIR, "batik_model_v3.tflite")
ONNX_PATH = os.path.join(OUTPUT_DIR, "batik_model_v3.onnx")
LABELS_PATH = os.path.join(OUTPUT_DIR, "labels.txt")
AUDIT_CSV_PATH = os.path.join(OUTPUT_DIR, "v3_dataset_audit.csv")
ACTIVE_LABELS_FILE = os.environ.get("BATIK_LABELS_FILE", "models/labels.txt")

# Image input dimensions
IMG_HEIGHT, IMG_WIDTH = 224, 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
IMG_CHANNELS = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# Training hyperparameters
BATCH_SIZE = 32
N_FOLDS = 5
TEST_SIZE = 0.15
PHASE1_EPOCHS = 20
PHASE2_EPOCHS = 40
PHASE1_LR = 1e-3
PHASE2_LR = 5e-5
PHASE1_PATIENCE = 5
PHASE2_PATIENCE = 7
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.3
MIN_LR = 1e-7

# Model architecture & loss hyperparameters
UNFREEZE_LAYERS = 120
DROPOUT_RATE_1 = 0.5
DROPOUT_RATE_2 = 0.3
DENSE_UNITS = 256
LOSS_MODE = "cce_smooth"
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05
EFFECTIVE_BETA = 0.999
MAX_CLASS_WEIGHT = 3.0
TTA_STEPS = 5

TIER1_THRESHOLD = 45
TIER2_THRESHOLD = 90

# GPU memory growth and mixed precision configuration
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU devices detected: {len(gpus)}")
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"Mixed precision policy set to: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("WARNING: No GPU detected. Execution will proceed on CPU.")

print("\nRuntime Configuration:")
print(f"  Platform         : {platform.platform()}")
print(f"  Python           : {platform.python_version()}")
print(f"  TensorFlow       : {tf.__version__}")
print("  Backbone         : EfficientNetV2S (v3/v4 hardened pipeline)")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Image dimensions : {IMG_SIZE}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Stratified Folds : {N_FOLDS}")
print(f"  Loss mode        : {LOSS_MODE} (label_smoothing={LABEL_SMOOTHING})")
print(f"  Unfreeze top N   : {UNFREEZE_LAYERS}")


## 2. Dataset Discovery, Deduplication, and Class Profiling


In [ ]:
"""Cell 2: Dataset discovery, 28-class filtering, dHash deduplication, quarantine, and clustering."""

import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.preprocessing import LabelEncoder


def load_active_labels_set(labels_file: str) -> set[str] | None:
    """Load the 28 active class names from labels.txt when available."""
    p = Path(labels_file)
    if not p.exists():
        return None
    return {line.strip() for line in p.read_text(encoding="utf-8").splitlines() if line.strip()}


def compute_dhash_int(image_path: str, hash_size: int = 8) -> int | None:
    """Compute difference hash (dHash) as a 64-bit integer, or None if unreadable."""
    try:
        with Image.open(image_path) as img:
            gray = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
            pixels = np.array(gray, dtype=np.int32)
            diff = pixels[:, 1:] > pixels[:, :-1]
            decimal_val = 0
            for bit in diff.flatten():
                decimal_val = (decimal_val << 1) | int(bit)
            return decimal_val
    except (OSError, UnidentifiedImageError, ValueError):
        return None


def hamming_distance(h1: int, h2: int) -> int:
    """Compute bitwise Hamming distance between two integers."""
    return (h1 ^ h2).bit_count()


def discover_dataset(root_dir: str, active_labels: set[str] | None = None) -> pd.DataFrame:
    """Discover all valid images grouped by class subdirectory."""
    valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    records = []
    root_path = Path(root_dir)

    if not root_path.exists():
        raise FileNotFoundError(f"Dataset directory not found: {root_dir}")

    class_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])
    if active_labels is not None:
        class_dirs = [d for d in class_dirs if d.name in active_labels]
    if not class_dirs:
        raise ValueError(f"No matching class subdirectories found in: {root_dir}")

    for class_dir in class_dirs:
        label = class_dir.name
        for filepath in sorted(class_dir.iterdir()):
            if filepath.is_file() and filepath.suffix.lower() in valid_extensions:
                records.append({"filepath": str(filepath), "label": label})

    return pd.DataFrame(records)


active_labels = load_active_labels_set(ACTIVE_LABELS_FILE)
if active_labels is not None:
    print(f"Filtering strictly to {len(active_labels)} active classes from {ACTIVE_LABELS_FILE}")
df = discover_dataset(DATASET_DIR, active_labels=active_labels)
print(f"Total raw images discovered: {len(df)}")

# Filter corrupted images and sub-resolution files
corrupted, dimension_issues = [], []
for idx, row in df.iterrows():
    img = cv2.imread(row["filepath"])
    if img is None:
        corrupted.append(idx)
        continue
    h, w = img.shape[:2]
    if h < 32 or w < 32:
        dimension_issues.append(idx)

if corrupted:
    df.drop(index=corrupted, inplace=True)
if dimension_issues:
    df.drop(index=dimension_issues, inplace=True)

df.reset_index(drop=True, inplace=True)
print(f"Valid images after integrity verification: {len(df)}")
print(f"  Corrupted files removed : {len(corrupted)}")
print(f"  Sub-resolution removed  : {len(dimension_issues)}")

# Perceptual hash scan with safe None handling and cross-class conflict quarantine
print("\nExecuting perceptual hash deduplication and cross-class quarantine scan...")
df["dhash"] = [compute_dhash_int(fp) for fp in df["filepath"]]
unhashable = df["dhash"].isna()
if unhashable.any():
    print(f"  Dropping {int(unhashable.sum())} unhashable images.")
    df = df.loc[~unhashable].reset_index(drop=True)

quarantined_cross_class: set[int] = set()
intra_class_drop: set[int] = set()
near_dup_edges: list[tuple[int, int]] = []

dhashes = df["dhash"].tolist()
labels_list = df["label"].tolist()
for i in range(len(df)):
    h1, c1 = dhashes[i], labels_list[i]
    for j in range(i + 1, len(df)):
        dist = hamming_distance(h1, dhashes[j])
        if dist <= 3:
            near_dup_edges.append((i, j))
        if dist <= 2:
            if c1 != labels_list[j]:
                # Quarantine BOTH images in cross-class conflicts (never bias alphabetically)
                quarantined_cross_class.add(i)
                quarantined_cross_class.add(j)
            elif i not in intra_class_drop and j not in intra_class_drop:
                intra_class_drop.add(j)

# Connected-component visual cluster IDs for GroupStratifiedKFold
parent = list(range(len(df)))


def _find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


for u, v in near_dup_edges:
    ru, rv = _find(u), _find(v)
    if ru != rv:
        parent[max(ru, rv)] = min(ru, rv)

df["cluster_id"] = [_find(i) for i in range(len(df))]

drop_indices = quarantined_cross_class | intra_class_drop
if drop_indices:
    print(f"  Quarantined cross-class conflict images (both sides): {len(quarantined_cross_class)}")
    print(f"  Auto-deduplicated intra-class images                : {len(intra_class_drop)}")
    df.drop(index=list(drop_indices), inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"Deduplicated dataset pool: {len(df)} images")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES = len(CLASS_NAMES)

with open(LABELS_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(CLASS_NAMES))

print(f"Saved class labels ({NUM_CLASSES} classes) to: {LABELS_PATH}")

# Class distribution analysis
class_counts = (
    df.groupby("label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

class_counts["tier"] = class_counts["count"].apply(
    lambda x: "Tier 1 (Aggressive)"
    if x <= TIER1_THRESHOLD
    else ("Tier 2 (Moderate)" if x <= TIER2_THRESHOLD else "Tier 3 (Light)")
)

tier_colors = {
    "Tier 1 (Aggressive)": "#d62728",
    "Tier 2 (Moderate)": "#ff7f0e",
    "Tier 3 (Light)": "#2ca02c",
}

fig, ax = plt.subplots(figsize=(14, 12))
sorted_counts = class_counts.sort_values("count", ascending=True)
bar_colors = [tier_colors[t] for t in sorted_counts["tier"]]
bars = ax.barh(sorted_counts["label"], sorted_counts["count"], color=bar_colors, edgecolor="white", height=0.75)
for bar, count in zip(bars, sorted_counts["count"]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, str(count), va="center", fontsize=8)

ax.axvline(x=TIER1_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.2, label=f"Tier 1 Threshold ({TIER1_THRESHOLD})")
ax.axvline(x=TIER2_THRESHOLD, color="#ff7f0e", linestyle="--", linewidth=1.2, label=f"Tier 2 Threshold ({TIER2_THRESHOLD})")
ax.set_title("Class Distribution and Augmentation Tier Profile (Batik v3/v4)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "v3_class_distribution.png"), dpi=150)
plt.close()


## 3. Stratified Holdout, Tier-Based Augmentations, and Data Pipelines


In [ ]:
"""Cell 3: Group-stratified holdout, domain-safe uint8 Albumentations, and tf.data loader."""

import albumentations as A
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# Fixed 15% test holdout (never touched during K-Fold training or validation)
X_trainval, X_test, y_trainval, y_test, g_trainval, g_test = train_test_split(
    df["filepath"].values,
    df["label_encoded"].values,
    df["cluster_id"].values,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label_encoded"].values,
)

print(f"Fixed test holdout pool : {len(X_test)} images")
print(f"Train/Validation pool   : {len(X_trainval)} images")

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Orientation-sensitive motifs (Parang diagonal polarity, Lontara script) must NOT be flipped
ORIENTATION_SENSITIVE_CLASSES = {
    "Solo_Parang",
    "Yogyakarta_Parang",
    "Sulawesi_Selatan_Lontara",
}


def build_tier_pipeline(tier: str, allow_hflip: bool = True) -> A.Compose:
    """Domain-safe uint8 Albumentations pipeline (no GridDistortion/ElasticTransform, bounded HSV)."""
    ops = [A.HorizontalFlip(p=0.5)] if allow_hflip else []
    if tier == "tier1":
        ops.extend([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=15, p=0.5),
            A.RandomResizedCrop(size=IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1), p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(16, 32), hole_width_range=(16, 32), p=0.25),
            A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.25),
            A.Sharpen(alpha=(0.1, 0.25), lightness=(0.85, 1.15), p=0.2),
        ])
    elif tier == "tier2":
        ops.extend([
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=15, p=0.4),
            A.RandomResizedCrop(size=IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.4),
            A.GaussianBlur(blur_limit=(3, 3), p=0.15),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
        ])
    else:
        ops.extend([
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
            A.HueSaturationValue(hue_shift_limit=4, sat_shift_limit=12, val_shift_limit=10, p=0.3),
        ])
    ops.append(A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH))
    return A.Compose(ops)


AUGMENT_PIPELINES = {
    (tier, allow_flip): build_tier_pipeline(tier, allow_hflip=allow_flip)
    for tier in ("tier1", "tier2", "tier3")
    for allow_flip in (True, False)
}

AUGMENT_VALIDATION = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TTA = build_tier_pipeline("tier3", allow_hflip=False)


def get_tier_for_class(class_name: str) -> str:
    count = df[df["label"] == class_name].shape[0]
    if count <= TIER1_THRESHOLD:
        return "tier1"
    if count <= TIER2_THRESHOLD:
        return "tier2"
    return "tier3"


CLASS_TIER_MAP = {i: get_tier_for_class(CLASS_NAMES[i]) for i in range(NUM_CLASSES)}
CLASS_ALLOW_FLIP_MAP = {i: CLASS_NAMES[i] not in ORIENTATION_SENSITIVE_CLASSES for i in range(NUM_CLASSES)}


def load_image_uint8(filepath: str) -> np.ndarray:
    """Load raw native-resolution RGB uint8 image (NEVER pre-resize or cast to float32)."""
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    if img is None:
        return np.zeros((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def augment_train(filepath: bytes, label: int) -> tuple:
    img_uint8 = load_image_uint8(filepath.numpy().decode("utf-8"))
    label_int = int(label.numpy())
    tier = CLASS_TIER_MAP.get(label_int, "tier2")
    allow_flip = CLASS_ALLOW_FLIP_MAP.get(label_int, True)
    img_aug = AUGMENT_PIPELINES[(tier, allow_flip)](image=img_uint8)["image"]
    return img_aug.astype(np.float32), label_int


def augment_eval(filepath: bytes, label: int) -> tuple:
    img_uint8 = load_image_uint8(filepath.numpy().decode("utf-8"))
    img_aug = AUGMENT_VALIDATION(image=img_uint8)["image"]
    return img_aug.astype(np.float32), int(label.numpy())


def augment_tta(filepath: bytes, label: int) -> tuple:
    img_uint8 = load_image_uint8(filepath.numpy().decode("utf-8"))
    img_aug = AUGMENT_TTA(image=img_uint8)["image"]
    return img_aug.astype(np.float32), int(label.numpy())


def _wrap(aug_fn):
    """Wrap Python numpy augmentation function for tf.data graph execution.

    Note that preprocess_input is intentionally OMITTED here: the model's
    internal Rescaling(1/127.5, offset=-1.0) layer processes raw [0, 255] float32.
    """
    def tf_fn(filepath, label):
        img, lbl = tf.py_function(func=aug_fn, inp=[filepath, label], Tout=[tf.float32, tf.int32])
        img.set_shape([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
        lbl.set_shape([])
        return img, tf.one_hot(lbl, depth=NUM_CLASSES)

    return tf_fn


tf_augment_train = _wrap(augment_train)
tf_augment_eval = _wrap(augment_eval)
tf_augment_tta = _wrap(augment_tta)

AUTOTUNE = tf.data.AUTOTUNE


def build_dataset(filepaths, labels, augment_fn, batch_size, shuffle=False, cache=False):
    """Construct optimized tf.data.Dataset pipeline."""
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds.batch(batch_size).prefetch(AUTOTUNE)


test_ds = build_dataset(X_test, y_test, tf_augment_eval, BATCH_SIZE, shuffle=False, cache=True)

print("Augmentation tier summary:")
for tier_name, tier_key in [("Tier 1 (Aggressive)", "tier1"), ("Tier 2 (Moderate)", "tier2"), ("Tier 3 (Light)", "tier3")]:
    classes_in_tier = [CLASS_NAMES[i] for i, t in CLASS_TIER_MAP.items() if t == tier_key]
    print(f"  {tier_name}: {len(classes_in_tier)} classes")


## 4. Model Architecture, Explicit Rescaling, and Focal Loss Formulation


In [ ]:
"""Cell 4: Backbone model architecture, calibrated loss, capped class weights, and cosine warmup."""

import math
from collections.abc import Sequence
from typing import Any

import numpy as np
import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="batik_v3")
class CategoricalFocalLoss(tf.keras.losses.Loss):
    """Categorical Focal Loss using true-class probability p_t for modulation."""

    def __init__(
        self,
        gamma: float = 2.0,
        alpha: Any = None,
        label_smoothing: float = 0.0,
        reduction: str = "sum_over_batch_size",
        name: str = "categorical_focal_loss",
    ):
        super().__init__(reduction=reduction, name=name)
        self.gamma = float(gamma)
        self.alpha = alpha
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_true_hard = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        p_t = tf.reduce_sum(y_true_hard * y_pred, axis=-1, keepdims=True)
        p_t = tf.clip_by_value(p_t, epsilon, 1.0 - epsilon)
        modulating_factor = tf.pow(1.0 - p_t, self.gamma)

        y_target = y_true_hard
        if self.label_smoothing > 0.0:
            num_classes = tf.cast(tf.shape(y_true_hard)[-1], tf.float32)
            y_target = y_true_hard * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)

        cross_entropy = -y_target * tf.math.log(y_pred)
        if self.alpha is not None:
            cross_entropy = tf.cast(self.alpha, tf.float32) * cross_entropy

        return tf.reduce_sum(modulating_factor * cross_entropy, axis=-1)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update({
            "gamma": self.gamma,
            "alpha": self.alpha,
            "label_smoothing": self.label_smoothing,
        })
        return config


def build_classification_loss(
    mode: str = "cce_smooth",
    label_smoothing: float = 0.05,
    gamma: float = 2.0,
    alpha: Any = None,
) -> tf.keras.losses.Loss:
    """Build v3/v4 classification loss ('cce_smooth' default for calibrated probabilities)."""
    if mode == "cce_smooth":
        return tf.keras.losses.CategoricalCrossentropy(label_smoothing=float(label_smoothing))
    if mode == "focal":
        return CategoricalFocalLoss(gamma=gamma, alpha=alpha, label_smoothing=label_smoothing)
    raise ValueError(f"Unsupported loss mode: {mode!r}")


def compute_effective_sample_weights(
    class_counts: Sequence[int],
    beta: float = 0.999,
    max_weight: float | None = 3.0,
) -> np.ndarray:
    """Compute capped Effective Number of Samples class weights."""
    counts = np.array(class_counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    effective_num = np.where(effective_num <= 0.0, 1e-8, effective_num)

    weights = (1.0 - beta) / effective_num
    normalized_weights = weights / np.sum(weights) * len(counts)
    if max_weight is not None:
        normalized_weights = np.minimum(normalized_weights, float(max_weight))
    return normalized_weights.astype(np.float32)


@tf.keras.utils.register_keras_serializable(package="batik_v4")
class CosineWarmupSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup followed by cosine decay down to min_lr."""

    def __init__(self, initial_lr: float, total_steps: int, warmup_steps: int, min_lr: float = 1e-7):
        super().__init__()
        self.initial_lr = float(initial_lr)
        self.total_steps = max(1, int(total_steps))
        self.warmup_steps = max(0, min(int(warmup_steps), self.total_steps))
        self.min_lr = float(min_lr)

    def __call__(self, step):
        step_f = tf.cast(step, tf.float32)
        warmup_f = tf.cast(self.warmup_steps, tf.float32)
        total_f = tf.cast(self.total_steps, tf.float32)
        init_lr = tf.cast(self.initial_lr, tf.float32)
        min_lr = tf.cast(self.min_lr, tf.float32)

        warmup_frac = tf.cond(
            warmup_f > 0.0,
            lambda: tf.minimum(step_f / tf.maximum(warmup_f, 1.0), 1.0),
            lambda: tf.constant(1.0, dtype=tf.float32),
        )
        warmup_lr = min_lr + (init_lr - min_lr) * warmup_frac

        decay_steps = tf.maximum(total_f - warmup_f, 1.0)
        decay_progress = tf.clip_by_value((step_f - warmup_f) / decay_steps, 0.0, 1.0)
        cosine_factor = 0.5 * (1.0 + tf.cos(tf.constant(math.pi, dtype=tf.float32) * decay_progress))
        decay_lr = min_lr + (init_lr - min_lr) * cosine_factor
        return tf.where(step_f < warmup_f, warmup_lr, decay_lr)

    def get_config(self) -> dict[str, Any]:
        return {
            "initial_lr": self.initial_lr,
            "total_steps": self.total_steps,
            "warmup_steps": self.warmup_steps,
            "min_lr": self.min_lr,
        }


def create_v3_backbone_model(
    num_classes: int = 28,
    input_shape: tuple[int, int, int] = (224, 224, 3),
    weights: str | None = "imagenet",
) -> tuple[tf.keras.Model, tf.keras.Model]:
    """Build EfficientNetV2S classifier with single explicit Rescaling layer."""
    inputs = tf.keras.Input(shape=input_shape, name="input_image", dtype=tf.float32)

    rescaled = tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1.0,
        name="input_rescaling",
    )(inputs)

    base_model = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights=weights,
        include_preprocessing=False,
        input_shape=input_shape,
    )

    features = base_model(rescaled)

    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(features)
    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE_1, name="head_dropout_1")(x)
    x = tf.keras.layers.Dense(DENSE_UNITS, activation="relu", name="head_dense")(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE_2, name="head_dropout_2")(x)
    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="batik_efficientnetv2s_v3")
    return base_model, model


print("Model architecture, calibrated loss, and cosine warmup schedule ready.")


## 5. Stratified 5-Fold Training Loop (Phase 1 & Phase 2 Fine-Tuning)


In [ ]:
"""Cell 5: StratifiedGroupKFold 5-Fold training loop with cosine warmup and safe checkpointing."""

import gc
import math
import os
import shutil

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import TopKCategoricalAccuracy
from tensorflow.keras.optimizers import Adam

fold_records = []
best_overall_val_acc = 0.0
best_overall_model_path = MODEL_PATH

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval, groups=g_trainval)):
    print(f"\n{'=' * 65}")
    print(f"  STARTING STRATIFIED GROUP FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'=' * 65}")

    X_tr, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

    tr_counts = [int(np.sum(y_tr == c)) for c in range(NUM_CLASSES)]
    class_weights_arr = compute_effective_sample_weights(
        tr_counts,
        beta=EFFECTIVE_BETA,
        max_weight=MAX_CLASS_WEIGHT,
    )
    fold_class_weights = dict(enumerate(class_weights_arr))

    train_ds_fold = build_dataset(X_tr, y_tr, tf_augment_train, BATCH_SIZE, shuffle=True)
    val_ds_fold = build_dataset(X_val, y_val, tf_augment_eval, BATCH_SIZE, shuffle=False)

    base_model, model = create_v3_backbone_model(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, weights="imagenet")

    # Phase 1: Train classification head with frozen backbone
    base_model.trainable = False
    loss_fn = build_classification_loss(
        mode=LOSS_MODE,
        label_smoothing=LABEL_SMOOTHING,
        gamma=FOCAL_GAMMA,
    )

    model.compile(
        optimizer=Adam(learning_rate=PHASE1_LR),
        loss=loss_fn,
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  [Fold {fold + 1}] Phase 1: Training classification head ({PHASE1_EPOCHS} epochs max)...")
    model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE1_EPOCHS,
        callbacks=[
            EarlyStopping(monitor="val_accuracy", patience=PHASE1_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
            ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=0),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    # Phase 2: Unfreeze top UNFREEZE_LAYERS (120); keep all BatchNorm layers frozen
    base_model.trainable = True
    for layer in base_model.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    fold_ckpt_path = os.path.join(OUTPUT_DIR, f"fold_{fold + 1}_best.keras")
    steps_per_epoch = max(1, math.ceil(len(X_tr) / BATCH_SIZE))
    total_p2_steps = steps_per_epoch * PHASE2_EPOCHS
    warmup_p2_steps = max(1, int(0.1 * total_p2_steps))
    p2_schedule = CosineWarmupSchedule(
        initial_lr=PHASE2_LR,
        total_steps=total_p2_steps,
        warmup_steps=warmup_p2_steps,
        min_lr=MIN_LR,
    )

    model.compile(
        optimizer=Adam(learning_rate=p2_schedule),
        loss=loss_fn,
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  [Fold {fold + 1}] Phase 2: Fine-tuning last {UNFREEZE_LAYERS} layers with cosine warmup ({PHASE2_EPOCHS} epochs max)...")
    history_p2 = model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE2_EPOCHS,
        callbacks=[
            ModelCheckpoint(filepath=fold_ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=0),
            EarlyStopping(monitor="val_accuracy", patience=PHASE2_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    fold_best_val_acc = max(history_p2.history["val_accuracy"])
    fold_best_val_top3 = max(history_p2.history["val_top3_accuracy"])

    # CRITICAL FIX (P1-3): Copy best checkpoint artifact instead of saving final epoch weights
    if fold_best_val_acc > best_overall_val_acc:
        best_overall_val_acc = fold_best_val_acc
        shutil.copy2(fold_ckpt_path, best_overall_model_path)
        print(f"  --> New overall best checkpoint copied: val_acc = {fold_best_val_acc:.4f}")

    fold_records.append({
        "fold": fold + 1,
        "val_accuracy": float(fold_best_val_acc),
        "val_top3_accuracy": float(fold_best_val_top3),
    })
    print(f"  Fold {fold + 1} Results -> Val Acc: {fold_best_val_acc:.4f} | Val Top-3: {fold_best_val_top3:.4f}")

    del model, base_model, train_ds_fold, val_ds_fold
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 65)
print("  5-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 65)
for rec in fold_records:
    print(f"  Fold {rec['fold']} -> Top-1: {rec['val_accuracy']:.4f} | Top-3: {rec['val_top3_accuracy']:.4f}")
mean_acc = np.mean([r["val_accuracy"] for r in fold_records])
print(f"\nMean Validation Accuracy across {N_FOLDS} folds: {mean_acc:.4f}")


## 6. Holdout Evaluation, Test-Time Augmentation (TTA), and Classification Report


In [ ]:
"""Cell 6: Test holdout evaluation, Test-Time Augmentation (TTA), and comprehensive metrics."""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print("Loading best model for test holdout evaluation...")
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"CategoricalFocalLoss": CategoricalFocalLoss},
)

# Standard evaluation on clean test set
print("Evaluating on clean test holdout...")
clean_eval = best_model.evaluate(test_ds, verbose=1)
print(f"Clean Test Holdout Loss    : {clean_eval[0]:.4f}")
print(f"Clean Test Holdout Accuracy: {clean_eval[1]:.4f}")

# Test-Time Augmentation (TTA)
print(f"\nExecuting {TTA_STEPS}-step Test-Time Augmentation (TTA)...")
tta_predictions = []

# Step 0: Clean pass
clean_preds = best_model.predict(test_ds, verbose=0)
tta_predictions.append(clean_preds)

# Steps 1..TTA_STEPS: Lightly augmented passes
for step in range(TTA_STEPS):
    tta_ds = build_dataset(X_test, y_test, tf_augment_tta, BATCH_SIZE, shuffle=False)
    preds = best_model.predict(tta_ds, verbose=0)
    tta_predictions.append(preds)

# Ensemble average
ensemble_probs = np.mean(tta_predictions, axis=0)
y_pred_classes = np.argmax(ensemble_probs, axis=1)

# Overall Accuracy Metrics
top1_acc = np.mean(y_pred_classes == y_test)
top3_hits = [y_test[i] in np.argsort(ensemble_probs[i])[-3:] for i in range(len(y_test))]
top3_acc = np.mean(top3_hits)

print("\n" + "=" * 65)
print("  FINAL TEST HOLDOUT METRICS (WITH TTA)")
print("=" * 65)
print(f"  Top-1 Accuracy : {top1_acc * 100:.2f}%")
print(f"  Top-3 Accuracy : {top3_acc * 100:.2f}%")

# Classification Report
target_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
clf_report = classification_report(y_test, y_pred_classes, target_names=target_names, digits=4)
print("\nPer-Class Classification Report:")
print(clf_report)

# Save report to disk
report_path = os.path.join(OUTPUT_DIR, "v3_classification_report.txt")
with open(report_path, "w") as f:
    f.write(clf_report)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(18, 16))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.title("Confusion Matrix (V3 Test Holdout with TTA)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "v3_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"Saved confusion matrix visualization to: {cm_path}")


## 7. Model Serialization (TFLite & ONNX FP16) and Parity Verification


In [ ]:
"""Cell 7: Model export to TFLite and ONNX (opset 17 + optional FP16), followed by parity validation."""

import os
import subprocess
import sys

import numpy as np
import tensorflow as tf

print("Step 1: Exporting FP16 TFLite model...")
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "CategoricalFocalLoss": CategoricalFocalLoss,
        "CosineWarmupSchedule": CosineWarmupSchedule,
    },
    compile=False,
)
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"Exported TFLite model: {TFLITE_PATH} ({tflite_size_mb:.2f} MB)")

print("\nStep 2: Exporting ONNX model via tf2onnx (opset 17)...")
onnx_cmd = [
    sys.executable,
    "-m",
    "tf2onnx.convert",
    "--keras",
    MODEL_PATH,
    "--output",
    ONNX_PATH,
    "--opset",
    "17",
]
try:
    subprocess.run(onnx_cmd, check=True)
    onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    print(f"Exported FP32 ONNX model: {ONNX_PATH} ({onnx_size_mb:.2f} MB)")

    # Optional FP16 ONNX quantization when onnxconverter_common is available
    try:
        import onnx
        from onnxconverter_common import float16

        fp16_onnx_path = ONNX_PATH.replace(".onnx", "_fp16.onnx")
        fp32_proto = onnx.load(ONNX_PATH)
        fp16_proto = float16.convert_float_to_float16(fp32_proto, keep_io_types=True)
        onnx.save(fp16_proto, fp16_onnx_path)
        fp16_size_mb = os.path.getsize(fp16_onnx_path) / (1024 * 1024)
        print(f"Exported FP16 ONNX model: {fp16_onnx_path} ({fp16_size_mb:.2f} MB)")
    except ImportError:
        print(
            "NOTICE: onnxconverter_common not installed; keeping FP32 ONNX artifact. "
            "Install `onnxconverter-common` for FP16 ONNX quantization."
        )
except Exception as e:
    print(f"ONNX conversion warning: {e}. Install tf2onnx (`pip install tf2onnx`).")

print("\nStep 3: Numerical Parity Verification across runtimes (batch_size=1 & batch_size=2)...")
for batch_size in (1, 2):
    dummy_input = np.random.uniform(0.0, 255.0, size=(batch_size, IMG_HEIGHT, IMG_WIDTH, 3)).astype(np.float32)
    keras_pred = best_model(dummy_input, training=False).numpy()

    if batch_size == 1:
        interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        interpreter.set_tensor(input_details[0]["index"], dummy_input)
        interpreter.invoke()
        tflite_pred = interpreter.get_tensor(output_details[0]["index"])
        max_diff_tflite = np.max(np.abs(keras_pred - tflite_pred))
        print(f"  [batch={batch_size}] Max abs deviation (Keras vs TFLite FP16): {max_diff_tflite:.6f}")
        assert max_diff_tflite < 1e-2, f"TFLite discrepancy too high: {max_diff_tflite}"

    if os.path.exists(ONNX_PATH):
        try:
            import onnxruntime as ort

            ort_sess = ort.InferenceSession(ONNX_PATH)
            onnx_input_name = ort_sess.get_inputs()[0].name
            onnx_pred = ort_sess.run(None, {onnx_input_name: dummy_input})[0]
            max_diff_onnx = np.max(np.abs(keras_pred - onnx_pred))
            print(f"  [batch={batch_size}] Max abs deviation (Keras vs ONNX): {max_diff_onnx:.6f}")
            assert max_diff_onnx < 1e-3, f"ONNX discrepancy too high: {max_diff_onnx}"
        except ImportError:
            print("  onnxruntime not installed; skipping ONNX runtime verification.")

print("\nModel serialization and parity verification complete.")
